# Emergency Analysis & Flight Path Map
Flags emergency squawk codes and renders interactive flight tracks with Folium.

## 1. Import & Load Data

In [16]:
import os
import sys
import pandas as pd
import folium
import numpy as np

# Robust project root detection
cwd = os.getcwd()
project_root = next(
    (os.path.abspath(p) for p in [cwd, os.path.join(cwd, '..'), os.path.join(cwd, '..', '..')]
     if os.path.exists(os.path.join(p, 'mapping.py'))),
    cwd
)
sys.path.insert(0, project_root)
from mapping import MILITARY_BASES

# Load data
df = pd.read_csv(os.path.join(project_root, 'data', 'aircraft_data_my_api.csv'))
df['datetime'] = pd.to_datetime(df['datetime'])
df['squawk'] = df['squawk'].astype(str).str.strip()
df = df.sort_values(['hex', 'datetime']).reset_index(drop=True)

print(f"Total records:     {len(df)}")
print(f"Unique aircraft:   {df['hex'].nunique()}")
print(f"Unique callsigns:  {df['callsign'].nunique()}")
print(f"Time range:        {df['datetime'].min()} → {df['datetime'].max()}")
df.head()

Total records:     406
Unique aircraft:   25
Unique callsigns:  13
Time range:        2026-05-25 23:28:46.034676 → 2026-05-25 23:52:18.778670


,hex,datetime,callsign,tail number,squawk,altitude,latitude,longitude,type,heading,ground speed,vertical rate,emergency,source
0,06a249,2026-05-25 23:28:46.034774,LHOB270,A7-MAB,2056,24000,36.708389,27.882500,C-17A Globemaster,335.0,448.0,-1280.0,False,adsb_icao
1,06a249,2026-05-25 23:29:47.456180,LHOB270,A7-MAB,2056,21700,36.820866,27.819078,C-17A Globemaster,335.0,432.0,-2208.0,False,adsb_icao
2,06a249,2026-05-25 23:30:48.984437,LHOB270,A7-MAB,2056,19500,36.933060,27.755848,C-17A Globemaster,335.0,425.0,-1824.0,False,adsb_icao
3,06a249,2026-05-25 23:31:50.315953,LHOB270,A7-MAB,2056,17875,37.043518,27.693670,C-17A Globemaster,335.0,414.0,-1408.0,False,adsb_icao
4,06a249,2026-05-25 23:32:51.687211,LHOB270,A7-MAB,2056,16200,37.148964,27.634138,C-17A Globemaster,335.0,408.0,-1632.0,False,adsb_icao


## 1b. Data Cleaning — Remove Spoofed / Erroneous Positions

In [17]:
from geopy.distance import geodesic as geo_distance

MAX_SPEED_KNOTS = 1500

def clean_track(ac_df):
    """Drop fixes that imply physically impossible speed from the previous accepted fix."""
    ac_df = ac_df.sort_values('datetime').reset_index(drop=True)
    keep = [0]  # always keep the first fix
    for i in range(1, len(ac_df)):
        prev = ac_df.iloc[keep[-1]]
        curr = ac_df.iloc[i]
        dt_hours = (curr['datetime'] - prev['datetime']).total_seconds() / 3600
        if dt_hours <= 0:
            continue
        dist_nm = geo_distance(
            (prev['latitude'], prev['longitude']),
            (curr['latitude'], curr['longitude'])
        ).nautical
        if dist_nm / dt_hours <= MAX_SPEED_KNOTS:
            keep.append(i)
    return ac_df.iloc[keep]


valid_mask = (
    pd.to_numeric(df['latitude'],  errors='coerce').notna() &
    pd.to_numeric(df['longitude'], errors='coerce').notna()
)
df_valid = df[valid_mask].copy()
df_valid['latitude']  = pd.to_numeric(df_valid['latitude'])
df_valid['longitude'] = pd.to_numeric(df_valid['longitude'])

before = len(df_valid)

# Use a for loop — avoids pandas 3.x groupby.apply dropping the key column
cleaned = [clean_track(grp) for _, grp in df_valid.groupby('hex')]
df_clean = pd.concat(cleaned, ignore_index=True) if cleaned else df_valid.iloc[0:0].copy()

removed = before - len(df_clean)
df_no_pos = df[~valid_mask]
df = pd.concat([df_clean, df_no_pos]).sort_values(['hex', 'datetime']).reset_index(drop=True)

print(f"Removed {removed} likely spoofed/erroneous fixes ({removed/before*100:.1f}% of positioned records)")
print(f"Remaining records: {len(df)}")

Removed 75 likely spoofed/erroneous fixes (18.5% of positioned records)
Remaining records: 331


## 2. Emergency Flag Analysis
Squawk codes: **7500** Hijacking · **7600** Radio Failure · **7700** General Emergency

In [18]:
EMERGENCY_SQUAWKS = {
    '7500': 'HIJACKING',
    '7600': 'RADIO FAILURE',
    '7700': 'GENERAL EMERGENCY'
}

# Primary: use the collector's emergency boolean; secondary: squawk cross-check
df['emergency_type'] = df['squawk'].map(EMERGENCY_SQUAWKS)
df['is_emergency'] = df['emergency'].astype(str).str.lower().isin(['true', '1']) | df['emergency_type'].notna()
# Fill emergency_type label for collector-flagged rows without a standard squawk
df.loc[df['is_emergency'] & df['emergency_type'].isna(), 'emergency_type'] = 'EMERGENCY (non-standard)'

emergencies = df[df['is_emergency']].copy()

if emergencies.empty:
    print('No emergencies detected in this dataset.')
else:
    print(f'⚠  EMERGENCIES DETECTED: {len(emergencies)} records\n')
    summary = emergencies.groupby(['callsign', 'type', 'squawk', 'emergency_type']).agg(
        first_seen=('datetime', 'min'),
        last_seen=('datetime', 'max'),
        records=('hex', 'count')
    ).reset_index()
    display(summary)

No emergencies detected in this dataset.


In [19]:
# Full emergency records detail
if not emergencies.empty:
    display(emergencies[['callsign', 'type', 'tail number',
                          'squawk', 'emergency_type', 'datetime',
                          'altitude', 'latitude', 'longitude']].reset_index(drop=True))
else:
    # Show squawk distribution as reference
    print('Top squawk codes observed in dataset:')
    display(df.groupby('squawk').size().sort_values(ascending=False).head(15).rename('count').reset_index())

Top squawk codes observed in dataset:


,squawk,count
0,Unavailable,33
1,1565,24
2,7442,24
3,4137,24
4,4131,24
5,0634,24
6,0700,23
7,0711,16
8,3416,15
9,4403,14


## 3. Flight Path Map
Each aircraft gets a colored track. Emergency squawkers are highlighted in red. Toggle military bases via the layer control.

In [20]:
TRACK_COLORS = [
    'cyan', 'lime', 'yellow', 'orange', 'deepskyblue',
    'magenta', 'lightgreen', 'coral', 'turquoise', 'gold',
    'violet', 'lightblue', 'greenyellow', 'tomato', 'aquamarine'
]

# Filter to rows with valid coordinates
map_df = df[~df['latitude'].astype(str).str.contains('UNKNOWN') &
            ~df['longitude'].astype(str).str.contains('UNKNOWN')].copy()
map_df['latitude'] = pd.to_numeric(map_df['latitude'], errors='coerce')
map_df['longitude'] = pd.to_numeric(map_df['longitude'], errors='coerce')
map_df = map_df.dropna(subset=['latitude', 'longitude', 'hex'])

print(f"Aircraft with position data: {map_df['hex'].nunique()}")
fix_counts = map_df.groupby('hex').size()
print(f"  ≥2 fixes (will draw track): {(fix_counts >= 2).sum()}")
print(f"  1 fix  (dot only):          {(fix_counts == 1).sum()}")

center_lat = map_df['latitude'].mean()
center_lon = map_df['longitude'].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=4,
    tiles='CartoDB dark_matter'
)

# --- Military Base Markers (toggleable layer) ---
base_group = folium.FeatureGroup(name='Military Bases', show=False)
for base_name, coords in MILITARY_BASES.items():
    folium.CircleMarker(
        location=[coords['lat'], coords['lon']],
        radius=4,
        color='#888888',
        fill=True,
        fill_color='#888888',
        fill_opacity=0.6,
        tooltip=base_name
    ).add_to(base_group)
base_group.add_to(m)

# --- Flight Paths (grouped by hex — one track per aircraft) ---
for i, (hex_id, ac_df) in enumerate(map_df.groupby('hex')):
    ac_df = ac_df.sort_values('datetime').reset_index(drop=True)
    if ac_df.empty:
        continue

    coords = list(zip(ac_df['latitude'], ac_df['longitude']))
    is_emergency = ac_df['is_emergency'].any()
    line_color = 'red' if is_emergency else TRACK_COLORS[i % len(TRACK_COLORS)]
    line_weight = 4 if is_emergency else 2

    first = ac_df.iloc[0]
    last  = ac_df.iloc[-1]
    callsign = last['callsign']
    label    = f"{callsign} ({hex_id})" if callsign == 'UNK C/S' else callsign
    ac_type  = last['type'] or '—'

    alt_vals = pd.to_numeric(ac_df['altitude'], errors='coerce').dropna()
    alt_range = f"{int(alt_vals.min()):,} – {int(alt_vals.max()):,} ft" if not alt_vals.empty else 'N/A'
    first_alt = f"{int(alt_vals.iloc[0]):,} ft"  if not alt_vals.empty else 'N/A'
    last_alt  = f"{int(alt_vals.iloc[-1]):,} ft" if not alt_vals.empty else 'N/A'

    gs_vals  = pd.to_numeric(ac_df['ground speed'], errors='coerce').dropna()
    gs_range = f"{int(gs_vals.min())} – {int(gs_vals.max())} kts" if not gs_vals.empty else 'N/A'

    emergency_badge = f'<br><b style="color:red">⚠ {last["emergency_type"]}</b>' if is_emergency else ''
    tooltip_text    = f"{label} | {ac_type} | {last_alt}"

    popup_html = f"""
    <div style="font-family: monospace; min-width: 210px; font-size: 12px">
        <b style="font-size: 14px">{label}</b>{emergency_badge}<br><br>
        <b>Type:</b> {ac_type}<br>
        <b>Tail #:</b> {last['tail number'] or '—'}<br>
        <b>Hex:</b> {hex_id}<br>
        <b>Squawk:</b> {last['squawk']}<br>
        <b>Altitude:</b> {alt_range}<br>
        <b>Speed:</b> {gs_range}<br>
        <b>Source:</b> {last.get('source', '—')}<br>
        <b>Fixes:</b> {len(ac_df)}<br>
        <b>First:</b> {str(ac_df['datetime'].min())[:19]}<br>
        <b>Last:</b> {str(ac_df['datetime'].max())[:19]}
    </div>
    """

    if len(coords) >= 2:
        folium.PolyLine(
            locations=coords,
            color=line_color,
            weight=line_weight,
            opacity=0.85,
            tooltip=tooltip_text
        ).add_to(m)

    # Start marker (always drawn)
    folium.CircleMarker(
        location=[first['latitude'], first['longitude']],
        radius=5,
        color='white',
        weight=1,
        fill=True,
        fill_color='green',
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{label} | {ac_type} | {first_alt} — start"
    ).add_to(m)

    # End marker (only when distinct from start)
    if len(coords) >= 2:
        end_fill = 'red' if is_emergency else 'white'
        folium.CircleMarker(
            location=[last['latitude'], last['longitude']],
            radius=5,
            color='white',
            weight=1,
            fill=True,
            fill_color=end_fill,
            fill_opacity=0.9,
            tooltip=f"{label} | {ac_type} | {last_alt} — last known"
        ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

output_path = os.path.join(project_root, 'analysis', 'flight_map.html')
m.save(output_path)
print(f'Map saved → {output_path}')

m

Aircraft with position data: 25
  ≥2 fixes (will draw track): 25
  1 fix  (dot only):          0
Map saved → /Users/jasonchristopher/Desktop/Code-Fellows/Projects/Aircraft_Tracker/analysis/flight_map.html
